In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path
import glob

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np

from src.load_data import load_confocal, load_confocal2

# Experimental conditions

In [3]:
experimental_path = Path.cwd().parent / 'data/silver_layer/experimental_results.pkl'
df_experimental = pd.read_pickle(experimental_path)

# Confocal

In [4]:
data_path = Path.cwd().parent / 'data/bronze_layer/confocal'

In [5]:
z1 = load_confocal(data_path, '2024_02_27', 12)
z2 = load_confocal(data_path, '2024_02_28', 12)
z3 = load_confocal(data_path, '2024_03_04', 12)
z4 = load_confocal(data_path, '2024_03_05', 12)
z5 = load_confocal(data_path, '2024_03_06', 12)
z6 = load_confocal(data_path, '2024_03_12', 12)
z7 = load_confocal(data_path, '2024_03_13', 12)
z8 = load_confocal(data_path, '2024_03_14', 12)
z9 = load_confocal(data_path, '2024_03_16', 14)
z10 = load_confocal(data_path, '2024_03_18', 14)
z11 = load_confocal(data_path, '2024_03_19', 14)
z12 = load_confocal(data_path, '2024_03_21', 14)
z13 = load_confocal(data_path, '2024_03_22', 14)
z14 = load_confocal(data_path, '2024_03_22_v2', 14)
z15 = load_confocal(data_path, '2024_03_23', 14)
z16 = load_confocal(data_path, '2024_03_25', 14)
z17 = load_confocal(data_path, '2024_03_25_v2', 14)
z18 = load_confocal(data_path, '2024_03_25_v3', 14)
z19 = load_confocal(data_path, '2024_03_26', 14)

In [6]:
z = []

z.extend([z1, z2, z3, z4, z5, z6, z7, z8, z9, z10, z11, z12, z13, z14, z15, z16, z17, z18, z19])

In [7]:
columns = ['t1_mean', 't1_median', 't1_std', 't1_missing_percentage', 't2_mean', 't2_median', 't2_std', 't2_missing_percentage', 'df', 'number', 'delta_time']

In [8]:
T1_THRESHOLD = 0.985
N_WATER = 1.333
N_BK7 = 1.519
MIN_T2 = 0.373
T2_THRESHOLDS = [8.02, 8.03, 8.04, 8.05, 8.10, 8.20, 8.50]

rows = []
run_id = 1

for j, zn in enumerate(z):

    for i, run_df in enumerate(zn):

        delta_time = (
            run_df['time'].diff().mean()
            if len(run_df) > 1 else np.nan
        )

        # Corrected t2 (d2 redistribution, no cap) for threshold statistics
        d1 = (run_df['t1'] - T1_THRESHOLD).clip(lower=0)
        d2 = d1 * (N_WATER / N_BK7)

        t2_corr = run_df['t2'].copy()
        mask_present = t2_corr.notna()
        t2_corr.loc[mask_present] = t2_corr.loc[mask_present] + d2[mask_present]
        mask_fill = t2_corr.isna() & (d2 > MIN_T2)
        t2_corr.loc[mask_fill] = d2[mask_fill]

        n_total = len(run_df)
        n_valid_t2 = int(t2_corr.notna().sum())

        row = {

            'run_id': run_id,

            't1_mean': round(run_df['t1'].mean(), 3),
            't1_median': round(run_df['t1'].median(), 3),
            't1_std': round(run_df['t1'].std(), 3),

            't1_missing_percentage':
                round(
                    100 - run_df['t1'].count()
                    / run_df['time'].count() * 100,
                    1
                ),

            't2_mean': round(run_df['t2'].mean(), 3),
            't2_median': round(run_df['t2'].median(), 3),
            't2_std': round(run_df['t2'].std(), 3),
            't2_max': round(run_df['t2'].max(), 3),

            't2_missing_percentage':
                round(
                    100 - run_df['t2'].count()
                    / run_df['time'].count() * 100,
                    1
                ),

            'delta_time': delta_time,
        }

        for thr in T2_THRESHOLDS:
            n_above = int((t2_corr > thr).sum())
            col = f'{thr:.2f}'.replace('.', '')
            row[f't2_above_{col}_pct_valid'] = (
                round(100 * n_above / n_valid_t2, 3) if n_valid_t2 > 0 else np.nan
            )
            row[f't2_above_{col}_pct_abs'] = round(100 * n_above / n_total, 3)

        rows.append(row)

        run_id += 1

df = pd.DataFrame(rows)

df['run_id'] = df['run_id'].astype(int)

In [9]:
save_path = Path.cwd().parent / 'data/silver_layer'
save_path.mkdir(parents=True, exist_ok=True)

df.to_pickle(
    save_path / 'confocal_results.pkl',
)

df.to_csv(
    save_path / 'confocal_results.csv'
)

In [10]:
save_path = Path.cwd().parent / 'data/silver_layer/confocal_runs'

save_path.mkdir(exist_ok=True)

run_id = 1

for zn in z:

    for run_df in zn:

        filename = save_path / f"confocal_run_{run_id}.csv"

        run_df.to_csv(filename, index=False)

        run_id += 1

# Process tube thickness

Before the tube-thickness correction, each run is trimmed so that it starts at the
first sample where the first thickness (`t1`, tube wall) is valid. Several runs
(68 of 164) begin with a long block where the sensor was not yet acquiring
(`t1` null, up to 96 % of the file); those rows carry no information and would
inflate the missing percentage of `t2`. After trimming, `time` is shifted so that
`time = 0` corresponds to the first valid `t1`.

In [11]:
T1_THRESHOLD = 0.985
N_WATER = 1.333
N_BK7 = 1.519
MIN_T2 = 0.373
T2_CAP = 8.02

source_path = Path.cwd().parent / 'data/silver_layer/confocal_runs'
processed_path = Path.cwd().parent / 'data/silver_layer/confocal_runs_processed'
processed_path.mkdir(exist_ok=True)

files = sorted(
    source_path.glob('confocal_run_*.csv'),
    key=lambda x: int(x.stem.split('_')[-1])
)

for file in files:
    run_df = pd.read_csv(file)

    # Start the run at the first valid t1 and reset time to 0 there
    first_valid = run_df['t1'].first_valid_index()

    if first_valid is None:
        print(f'{file.name}: no valid t1, kept untrimmed')
    else:
        run_df = run_df.loc[first_valid:].reset_index(drop=True)
        run_df['time'] = run_df['time'] - run_df['time'].iloc[0]

    d1 = (run_df['t1'] - T1_THRESHOLD).clip(lower=0)
    d2 = d1 * (N_WATER / N_BK7)

    run_df['t1'] = run_df['t1'] - d1

    mask_present = run_df['t2'].notna()
    run_df.loc[mask_present, 't2'] = run_df.loc[mask_present, 't2'] + d2[mask_present]

    mask_fill = run_df['t2'].isna() & (d2 > MIN_T2)
    run_df.loc[mask_fill, 't2'] = d2[mask_fill]

    run_df['t2'] = run_df['t2'].clip(upper=T2_CAP)

    run_df.to_csv(processed_path / file.name, index=False)

In [12]:
confocal_path = (
    Path.cwd().parent
    / "data/silver_layer/confocal_runs_processed"
)

rows = []

# Sort by run number if files are named confocal_run_1.csv, confocal_run_2.csv, ...
files = sorted(
    confocal_path.glob("confocal_run_*.csv"),
    key=lambda x: int(x.stem.split("_")[-1])
)

for run_id, file in enumerate(files, start=1):

    run_df = pd.read_csv(file)

    delta_time = (
        run_df["time"].diff().mean()
        if len(run_df) > 1 else np.nan
    )

    rows.append({

        "run_id": run_id,

        "t1_mean": round(run_df["t1"].mean(), 3),
        "t1_median": round(run_df["t1"].median(), 3),
        "t1_std": round(run_df["t1"].std(), 3),

        "t1_missing_percentage":
            round(
                100
                - run_df["t1"].count()
                / run_df["time"].count() * 100,
                1
            ),

        "t2_mean": round(run_df["t2"].mean(), 3),
        "t2_median": round(run_df["t2"].median(), 3),
        "t2_std": round(run_df["t2"].std(), 3),

        "t2_missing_percentage":
            round(
                100
                - run_df["t2"].count()
                / run_df["time"].count() * 100,
                1
            ),

        "delta_time": delta_time,

        "n_samples": len(run_df),
        "duration": round(run_df["time"].iloc[-1] - run_df["time"].iloc[0], 3),
    })

df = pd.DataFrame(rows)

df["run_id"] = df["run_id"].astype(int)

In [13]:
save_path = Path.cwd().parent / 'data/silver_layer'
save_path.mkdir(parents=True, exist_ok=True)

df.to_pickle(
    save_path / 'confocal_results_processed.pkl',
)

df.to_csv(
    save_path / 'confocal_results_processed.csv'
)

# Remove mixer waves (processed_v2)

In separated flows the liquid accumulated at the mixer is released as short waves that almost fill the channel. At the probe they appear as isolated tall readings ($t_2$ of 6 mm to 8 mm) lasting 1 ms to 3 ms inside a thin film. Real liquid slugs and plugs last 15 ms to 130 ms, and the liquid of bubbly flow is surrounded by other tall readings.

Starting from `confocal_runs_processed`, an event ($t_2 \geq 4$ mm, readings closer than 20 ms grouped together) is removed only when it is:

* **short**: its longest contiguous tall stretch lasts less than 5 ms
* **inside a film**: the median valid $t_2$ within 0.2 s before and after it is below 2 mm
* **isolated**: less than 10 % of the readings within 0.2 s around it are tall

The removed samples (including their rise and fall above 2 mm) get $t_2$ = NaN and are flagged in the `mixer_wave` column, so the analysis can exclude them instead of counting them as undetected. At 100 Hz one sample lasts 10 ms, so the duration test cannot be met and those runs are not changed: mixer waves and short slugs cannot be told apart at that rate. Parameters: `MIXER_WAVE_PARAMS` in `src/confocal_treatment.py`.

In [14]:
from src.confocal_treatment import remove_mixer_waves, MIXER_WAVE_PARAMS

processed_path = Path.cwd().parent / 'data/silver_layer/confocal_runs_processed'
processed_v2_path = Path.cwd().parent / 'data/silver_layer/confocal_runs_processed_v2'
processed_v2_path.mkdir(exist_ok=True)

files = sorted(
    processed_path.glob('confocal_run_*.csv'),
    key=lambda x: int(x.stem.split('_')[-1])
)

removal_rows = []

for file in files:

    run_df = remove_mixer_waves(pd.read_csv(file))

    run_df.to_csv(processed_v2_path / file.name, index=False)

    removal_rows.append({
        'run_id': int(file.stem.split('_')[-1]),
        'n_samples': len(run_df),
        'n_removed': int(run_df['mixer_wave'].sum()),
        'removed_percentage': round(100 * run_df['mixer_wave'].mean(), 3),
    })

df_removal = pd.merge(
    pd.DataFrame(removal_rows),
    df_experimental[['run_id', 'jl', 'jg', 'pattern', 'reduced_pattern']],
    on='run_id',
    how='left',
)

df_removal.to_csv(Path.cwd().parent / 'data/silver_layer/mixer_wave_removal.csv', index=False)

df_removal.groupby('reduced_pattern').agg(
    runs=('run_id', 'size'),
    runs_affected=('n_removed', lambda s: int((s > 0).sum())),
    removed_samples=('n_removed', 'sum'),
)

,runs,runs_affected,removed_samples
reduced_pattern,,,
annular,11,0,0
bubbly,11,0,0
churn,14,1,7
dispersed,20,0,0
plug,33,2,15
slug,25,6,1406
stratified,7,3,1228
wavy,43,9,2731


In [15]:
# Summary of processed_v2 (same columns as confocal_results_processed)
rows = []

files_v2 = sorted(
    processed_v2_path.glob('confocal_run_*.csv'),
    key=lambda x: int(x.stem.split('_')[-1])
)

for file in files_v2:

    run_df = pd.read_csv(file)
    kept = run_df.loc[~run_df['mixer_wave']]

    rows.append({
        'run_id': int(file.stem.split('_')[-1]),

        't1_mean': round(kept['t1'].mean(), 3),
        't1_median': round(kept['t1'].median(), 3),
        't1_std': round(kept['t1'].std(), 3),
        't1_missing_percentage': round(100 - kept['t1'].count() / kept['time'].count() * 100, 1),

        't2_mean': round(kept['t2'].mean(), 3),
        't2_median': round(kept['t2'].median(), 3),
        't2_std': round(kept['t2'].std(), 3),
        't2_missing_percentage': round(100 - kept['t2'].count() / kept['time'].count() * 100, 1),

        'delta_time': run_df['time'].diff().mean() if len(run_df) > 1 else np.nan,
        'n_samples': len(run_df),
        'n_mixer_wave': int(run_df['mixer_wave'].sum()),
        'duration': round(run_df['time'].iloc[-1] - run_df['time'].iloc[0], 3),
    })

df_v2 = pd.DataFrame(rows)

df_v2.to_pickle(Path.cwd().parent / 'data/silver_layer/confocal_results_processed_v2.pkl')
df_v2.to_csv(Path.cwd().parent / 'data/silver_layer/confocal_results_processed_v2.csv')

df_removal.loc[df_removal['n_removed'] > 0].sort_values(['reduced_pattern', 'n_removed'])

,run_id,n_samples,n_removed,removed_percentage,jl,jg,pattern,reduced_pattern
25,26,10000,7,0.070,0.81,5.43,churn,churn
35,36,10000,6,0.060,0.24,1.34,plug,plug
34,35,10000,9,0.090,0.12,1.31,plug,plug
23,24,10000,36,0.360,0.43,6.11,slug,slug
15,16,10000,46,0.460,0.48,6.50,slug,slug
14,15,10000,143,1.430,0.49,3.33,slug,slug
22,23,10000,252,2.520,0.61,3.44,slug,slug
13,14,10000,387,3.870,0.61,2.79,slug,slug
24,25,10000,542,5.420,0.86,2.66,slug,slug
47,48,10000,196,1.960,0.05,5.33,stratified,stratified


# Merge experimental

In [16]:
df_combined = pd.merge(df, df_experimental[['run_id', 'jl', 'jg', 'date_identifier', 'pattern', 'reduced_pattern']], on='run_id', how='left')

In [17]:
pd.read_pickle(save_path/'confocal_results_processed.pkl')['t2_mean'].sum()

np.float64(346.847)

In [18]:
pd.read_pickle(save_path/'confocal_results.pkl')['t2_mean'].sum()

np.float64(347.508)